# 🧰 ISOM 260: Same Model, Different Harness

A startup posts an AI tool that turns a photo into a 3D world. Someone replies: *"isn't this built with a simple prompt?"* The founder's answer: the prompt is simple; the code and instructions **around** the model are not. That wrapper is the **harness**.

Today you test that claim yourself: **same model, same photo, three harnesses.** The job is the first step of [image-blaster](https://github.com/harslan/image-blaster), an open-source tool built on Claude: look at a photo and list the objects worth turning into 3D models. Each object on that list becomes a paid 3D generation, so every bad line costs money.

| Level | The harness | What you'll see |
|---|---|---|
| 0 | just ask | a friendly paragraph-y list |
| 1 | + the rules image-blaster actually uses | a cleaner list, in JSON |
| 2 | + a check written in code, and one chance to fix | a list you can trust, and proof of it |

**🎲 Bet before Level 0 runs:** of the objects Level 0 lists, how many would a 3D pipeline reject? Write the number down.

## 🚀 Part 1: Setup (1 minute)

Only `GOOGLE_API_KEY`, same as every session.

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

from google import genai
from google.genai import types
import json, re, urllib.request

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL  = "gemini-2.5-flash"      # a brand-new key may be told 2.5 is "not found" → use "gemini-3.8-flash" (also free)
IN_PRICE, OUT_PRICE = 0.30, 2.50 # $ per 1M tokens, Gemini 2.5 Flash paid tier
print("✅ Gemini ready")

## 📸 Part 2: The photo

The class photo is a student's study desk: laptop, notebook, pens, sticky notes, books, a lamp. (Photo by [Yen Vu](https://unsplash.com/photos/desk-with-open-book-laptop-and-study-materials-HNjWq8WPyoY) on Unsplash, free under the Unsplash License. Its own caption calls it "a cozy late-night study workspace" — remember that word.)

Want to use your own? Set `USE_MY_OWN = True` and upload one: your desk, your bag, a kitchen counter.

In [ ]:
USE_MY_OWN = False
PHOTO_URL  = "https://images.unsplash.com/photo-1769794371055-54436b54577e?w=1600&q=80&fm=jpg"

if USE_MY_OWN:
    from google.colab import files
    up = files.upload()
    PHOTO = next(iter(up.values()))
else:
    try:
        PHOTO = urllib.request.urlopen(PHOTO_URL, timeout=20).read()
    except Exception as e:
        print(f"Couldn't fetch the class photo ({e}). Upload one instead:")
        from google.colab import files
        up = files.upload()
        PHOTO = next(iter(up.values()))

def mime_of(b):
    if b[:4] == b"\x89PNG": return "image/png"
    if b[:4] == b"RIFF" and b[8:12] == b"WEBP": return "image/webp"
    if b[4:8] == b"ftyp" and b[8:12] in (b"heic", b"heix", b"mif1", b"msf1"): return "image/heic"   # iPhone photos
    return "image/jpeg"
MIME = mime_of(PHOTO)
from IPython.display import Image, display
try:
    display(Image(data=PHOTO, width=520))
except Exception:
    print(f"📷 photo loaded ({len(PHOTO)//1024} KB, {MIME}) — Colab can't preview this format, but Gemini can read it")

## 🛠️ Part 3: Two helpers

`ask()` sends the photo plus a prompt and keeps the bill. `check()` is the harness's code check. It reads each entry the way a 3D pipeline would, which makes **one model per entry**, and rejects:

- **two things in one entry** ("mug and saucer" becomes one blob mesh)
- **many things in one entry** ("books" is a pile, not a book: list one book with a count)
- **the same kind of thing twice** (a red pen and a black pen are one pen model, × 2: color is paint, not shape)
- **part of the room** (you can't lift a window)
- **mood words** ("a cozy mug" is a vibe, not a shape)

It never calls the model, so it gives the same answer every time.

In [ ]:
BILL = {}

def ask(level, prompt, system=None, as_json=False):
    cfg = types.GenerateContentConfig(system_instruction=system,
                                      response_mime_type="application/json" if as_json else None)
    r = client.models.generate_content(model=MODEL, config=cfg,
        contents=[types.Part.from_bytes(data=PHOTO, mime_type=MIME), prompt])
    u = r.usage_metadata
    b = BILL.setdefault(level, {"calls": 0, "in": 0, "out": 0})
    b["calls"] += 1
    b["in"]  += u.prompt_token_count or 0
    b["out"] += (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
    return r.text or ""

COLORS = set("red white blue black green yellow orange purple pink brown gray grey silver gold beige clear dark light navy teal".split())
DESCRIPTIVE = set("writing text notes note label labels logo logos pattern print design stickers sticker lid cap handle cover artwork photo picture drawing stripes".split())
STATE = set("open closed stacked standing loose folded empty full small large big tiny partially visible single spare".split())
NOT_PLURAL = set("glasses scissors headphones earbuds pants jeans shorts lens canvas bus series species chess dress mattress compass grass".split())
GROUPED  = re.compile(r"\b(set|pile|stack|group|collection|bunch|pair|row) of\b|&|\bplus\b", re.I)
SURFACE  = re.compile(r"\b(wall|walls|floor|ceiling|window|windows|rug|carpet|room|background|lighting|light from|shadow|view|pinboard|corkboard|shelf|shelves)\b", re.I)
MOOD     = re.compile(r"\b(cozy|mysterious|lonely|inviting|feels?|suggests?|hints?|vibe|atmosphere|charming|cluttered|busy|aesthetic|messy)\b", re.I)

def core_of(name):
    return re.sub(r"\s*\([^)]*\)", "", name).strip().lower()

def singular(w):
    if w in NOT_PLURAL or w.endswith("ss") or len(w) < 4: return w
    if w.endswith("ies"): return w[:-3] + "y"
    if w.endswith(("ches", "shes", "xes")): return w[:-2]
    return w[:-1] if w.endswith("s") else w

def two_things(core):
    if GROUPED.search(core): return True
    for a, b in re.findall(r"(\w+) and (\w+)", core):
        if not (a in COLORS and b in COLORS): return True
    m = re.search(r"\bwith (\w+)", core)
    return bool(m and m.group(1) not in DESCRIPTIVE)

def check(objects):
    """objects: list of {'name', optional 'count', 'note'}. Returns rows of (name, [problems])."""
    rows, seen = [], {}
    for o in objects:
        name = (o.get("name") or "").strip()
        core = core_of(name)
        main = re.split(r"\bwith\b", core)[0]                        # "laptop with stickers" → laptop
        words = [w for w in re.findall(r"[a-z]+", main) if w not in ("and", "a", "an", "the", "of")]
        last = words[-1] if words else ""
        head = singular(last)
        kind = " ".join(singular(w) for w in words if w not in COLORS and w not in STATE)
        problems = []
        if two_things(core):                       problems.append("two things in one entry (one blob mesh)")
        if "count" not in o and last != head:      problems.append("many things in one entry (list one, with a count)")
        if SURFACE.search(core):                   problems.append("part of the room, not something you could lift")
        if MOOD.search(f'{name} {o.get("note", "")}'): problems.append("mood word, not a description")
        if kind and kind in seen:                  problems.append(f"same kind of thing as '{seen[kind]}' (one model, use a count)")
        seen.setdefault(kind, name)
        rows.append((name, problems))
    return rows

def report(rows):
    bad = sum(1 for _, p in rows if p)
    for name, p in rows:
        print(("❌ " if p else "✅ ") + name + ("  ← " + "; ".join(p) if p else ""))
    print(f"\n{bad} of {len(rows)} would be rejected.")
    return bad

print("✅ helpers ready")

## 0️⃣ Part 4: Level 0 — just ask

Commit to your bet first.

In [ ]:
L0 = ask(0, "List the objects in this photo that we could turn into 3D models, one per line.")
print(L0, "\n" + "─" * 60)

def parse_lines(text):
    """One object per line; split 'Name: note' or 'Name - note'; skip an intro line ending in ':'."""
    out = []
    for line in text.splitlines():
        line = re.sub(r"^\s*([-*•]|\d+[.)])\s*", "", line).replace("**", "").strip()
        if not line or line.endswith(":"):
            continue
        parts = re.split(r":\s|\s[–—-]\s", line, maxsplit=1)
        out.append({"name": parts[0].strip(), "note": parts[1].strip() if len(parts) > 1 else ""})
    return out

L0_objects = parse_lines(L0)
L0_bad = report(check(L0_objects))

## 1️⃣ Part 5: Level 1 — the rules a real product uses

These four rules are condensed from image-blaster's own instruction file ([`IMAGE-BLAST.md`](https://github.com/harslan/image-blaster/blob/main/.claude/skills/image-blast-uncover/IMAGE-BLAST.md), MIT license). Nothing else changes: same model, same photo.

In [ ]:
RULES = """You are surveying a photo for a 3D asset pipeline.
1. Describe like a technical scene survey: concrete, visible facts only. No mood or story words.
2. Include only single items a person could lift or push. Never walls, floors, rugs, windows or the room.
3. Never group different items into one entry ("desk with laptop" is two entries: desk, laptop).
4. If the same item appears several times, list it once with a count.
Return JSON: {"objects": [{"name": str, "count": int, "material": str, "note": str}]}"""

L1 = json.loads(ask(1, "Survey this photo.", system=RULES, as_json=True))["objects"]
for o in L1: print(f'{o.get("count", 1)} × {o["name"]}  ({o.get("material", "")})')
print("─" * 60)
L1_bad = report(check(L1))

## 2️⃣ Part 6: Level 2 — a hook, not a hope

Rules make good output **likely**. Code makes it **checked**. If anything fails, the failures go back to the model once with the exact reason, the same move as Monday's critic. Two rules from Monday still hold: the checker never writes, and the loop stops after one fix.

In [ ]:
rows = check(L1)
failed = [(n, p) for n, p in rows if p]
L2 = L1
if failed:
    fix = "These entries failed the pipeline check:\n" + "\n".join(f"- {n}: {'; '.join(p)}" for n, p in failed)
    fix += "\nReturn the full corrected JSON list. Same rules."
    L2 = json.loads(ask(2, fix, system=RULES, as_json=True))["objects"]
    print(f"🔁 sent {len(failed)} entr{'y' if len(failed) == 1 else 'ies'} back once\n")
else:
    print("Nothing failed — the check still ran, and that is the point.\n")
L2_bad = report(check(L2))

## 💵 Part 7: The bill, and the boardroom

Each listed object becomes one paid 3D generation. A rejected line is money spent on a mesh nobody can use.

In [ ]:
def cost(b): return (b["in"] * IN_PRICE + b["out"] * OUT_PRICE) / 1e6
total = {"calls": 0, "in": 0, "out": 0}
print(f'{"Level":<8}{"calls":>6}{"tokens in":>11}{"tokens out":>12}{"$ / photo":>12}{"× 1,000 photos":>16}')
for lvl in (0, 1, 2):
    b = BILL.get(lvl, {"calls": 0, "in": 0, "out": 0})
    if lvl == 2:                                     # Level 2 = Level 1's call + the fix (if any)
        b = {k: BILL.get(1, {}).get(k, 0) + b.get(k, 0) for k in ("calls", "in", "out")}
    print(f'{lvl:<8}{b["calls"]:>6}{b["in"]:>11,}{b["out"]:>12,}{cost(b):>12.4f}{cost(b) * 1000:>16.2f}')
print(f"\nRejected objects: Level 0 → {L0_bad}, Level 1 → {L1_bad}, Level 2 → {L2_bad}")

**For the boardroom (one sentence each):**
1. The model is the same at every level. What did you actually pay for between Level 0 and Level 2?
2. Your midterm agent: name **one rule** to write down, **one job** to move into code, and **one human gate**. Those three go in your HW #9 pitch.